## ⚔️ ACLED Collection — Colombian Conflict & Civic Space Events

### By:
MGO

### Date:
2026-03-29

### Description:

Collect geocoded conflict and protest event data from ACLED (Armed Conflict Location
& Event Data) for Colombia. ACLED is the gold-standard source for ground-truth offline
events — protests, violence against civilians, state repression — that we correlate with
online discourse from RSS and GDELT.

**Why we filter by keywords:** This project's research scope is *electoral integrity* and
*civic space health*, not all armed conflict. By filtering ACLED notes for election-related
and civic space keywords, we focus on events that directly connect to the 2026 election
cycle — e.g., attacks on social leaders who oppose candidates, ESMAD interventions during
electoral protests, or threats against journalists covering the campaign.

**Key event types collected:**
- **Protests** — peaceful protests, protests with intervention, excessive force
- **Riots** — mob violence, property destruction
- **Violence against civilians** — attacks on social leaders, human rights defenders
- **Strategic developments** — political agreements, policy changes

**API:** ACLED REST API (requires registration at https://acleddata.com — 24-48h approval)

**Data source priority:** P1 (Day 2)

**Output:** `data/01_raw/acled/acled_events_YYYYMMDD.json`

## 📚 Import libraries

In [ ]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

# Add src to path for importing project utilities
sys.path.insert(0, str(Path("../../src").resolve()))

from collectors.acled import query_acled
from utils.config import load_config

# Load environment variables (.env contains ACLED_EMAIL and ACLED_PASSWORD)
load_dotenv(Path("../../.env"))

sns.set_theme(style="darkgrid")

## ⚙️ Configuration

Load ACLED filter settings and research keywords from YAML config files.
The `notes_keywords` list combines:
1. ACLED-specific civic space terms (from `acled.yml`) — social leaders, journalists, etc.
2. Election keywords (from `election.yml`) — to catch events tied to the 2026 cycle
3. Civic space keywords (from `civic_space.yml`) — protests, repression, press freedom

This keyword filter runs **in-memory after the API call**, keeping our dataset focused
on the research scope without missing raw data that ACLED may classify differently.

In [ ]:
from __future__ import annotations

# ACLED collection configuration
acled_config = load_config("../../conf/data_collection/acled.yml")
acled_filters = acled_config["filters"]
acled_settings = acled_config["settings"]

# Election and civic space keyword lists (our research scope)
election_config = load_config("../../conf/keywords/election.yml")
civic_config = load_config("../../conf/keywords/civic_space.yml")

# Combine all notes_keywords: ACLED-specific + election + civic_space
# We do NOT include electoral hashtags (they don't appear in ACLED event notes)
notes_keywords: list[str] = (
    acled_settings["notes_keywords"]
    + election_config["keywords"]
    + civic_config["keywords"]
)
# Deduplicate while preserving order
seen: set[str] = set()
notes_keywords = [kw for kw in notes_keywords if not (kw in seen or seen.add(kw))]  # type: ignore[func-returns-value]

print("ACLED Configuration:")
print(f"  Country: {acled_filters['country']}")
print(f"  Event types: {acled_filters['event_types']}")
print(f"  Lookback days: {acled_settings['lookback_days']}")
print(f"  API limit (per page): {acled_settings['limit']}")
print(f"\nTotal notes_keywords for post-filtering: {len(notes_keywords)}")
print(f"  Sample keywords: {notes_keywords[:10]}")

# NOTE: ACLED database lags behind real-time by several months.
# The most recent available Colombia data is from late-2024 through early-2025.
# We use a fixed 90-day window anchored to the last known data cutoff (2025-03-29)
# rather than datetime.now() — because this project runs in a simulated 2026 environment
# but ACLED only has real data through ~2025-03.
# This window captures the pre-election monitoring period: Dec 2024 - Mar 2025.
ACLED_DATA_END_DATE = "2025-03-29"   # last confirmed available date in ACLED Colombia
ACLED_WINDOW_DAYS = 90               # 3-month research window
from datetime import datetime as _dt, timedelta as _td  # noqa: E402  (local alias)
_end = _dt.strptime(ACLED_DATA_END_DATE, "%Y-%m-%d")
start_date = (_end - _td(days=ACLED_WINDOW_DAYS)).strftime("%Y-%m-%d")
end_date = ACLED_DATA_END_DATE
today_str = _dt.now().strftime("%Y%m%d")  # file timestamp (uses system clock)
print(f"\nDate range: {start_date} to {end_date} (fixed 90-day ACLED window)")
print(f"Note: ACLED data availability ends ~2025-03 — dynamic datetime.now() would return empty results")

# Check ACLED API credentials
has_acled_credentials = bool(os.getenv("ACLED_EMAIL") and os.getenv("ACLED_PASSWORD"))
print(f"\nACLED API credentials configured: {has_acled_credentials}")
if not has_acled_credentials:
    print(
        "NOTE: Register at https://acleddata.com for API access (24-48h approval).\n"
        "Set ACLED_EMAIL and ACLED_PASSWORD in your .env file.\n"
        "This notebook will use synthetic sample data until credentials are available."
    )

## 💾 Data Collection

Query the ACLED API for Colombian conflict/protest events within the configured
lookback window. We pass:
- `event_types` — from config (Protests, Riots, Violence against civilians, Strategic developments)
- `date_range` — computed dynamically from `lookback_days` to today
- `notes_keywords` — the combined keyword list assembled above

If ACLED credentials are not yet configured, we fall back to a synthetic sample
that demonstrates the schema and lets downstream cells run without errors.

In [ ]:
from __future__ import annotations

if has_acled_credentials:
    try:
        # NOTE: notes_keywords is set to None here because ACLED notes are written in ENGLISH
        # while our keyword lists are in Spanish. Keyword post-filtering with Spanish terms
        # would reduce results to near-zero (< 10 events from 1200+).
        # Research-scope filtering is handled by event_types (Protests, Riots,
        # Violence against civilians, Strategic developments) — this is the correct
        # primary filter. Keyword filtering is applied at the DuckDB load stage where
        # English civic-space terms are used.
        df_acled = query_acled(
            country=acled_filters["country"],
            event_types=acled_filters["event_types"],
            date_range=(start_date, end_date),
            notes_keywords=None,  # disabled — notes are English, keywords are Spanish
            limit=acled_settings["limit"],
        )
        print(f"Collected {len(df_acled)} events from ACLED (live API)")
    except Exception as e:
        print(f"ACLED query failed: {e}")
        print("Falling back to empty DataFrame — check credentials and API status")
        df_acled = pd.DataFrame()
else:
    # Synthetic sample data demonstrating the ACLED schema.
    # Events span the 5 core departments with highest civic tension in Colombia.
    print("Using synthetic sample data (set ACLED_EMAIL + ACLED_PASSWORD for live data)")
    df_acled = pd.DataFrame(
        [
            {
                "event_id": "ACL001",
                "event_date": "2026-03-10",
                "event_type": "Protests",
                "sub_event_type": "Peaceful protest",
                "disorder_type": "Political violence",
                "actor1": "Students (Colombia)",
                "actor2": "",
                "assoc_actor_1": "",
                "assoc_actor_2": "",
                "inter1": "6",
                "location": "Bogota",
                "latitude": 4.711,
                "longitude": -74.0721,
                "geo_precision": 1,
                "admin1": "Bogota",
                "admin2": "",
                "civilian_targeting": "",
                "fatalities": 0,
                "notes": "Protesta estudiantil pacifica en la Plaza de Bolivar exigiendo garantias electorales",
                "source": "Local media",
                "tags": "elecciones",
            },
            {
                "event_id": "ACL002",
                "event_date": "2026-03-12",
                "event_type": "Violence against civilians",
                "sub_event_type": "Attack",
                "disorder_type": "Political violence",
                "actor1": "Unidentified Armed Group",
                "actor2": "Civilians (Colombia)",
                "assoc_actor_1": "",
                "assoc_actor_2": "",
                "inter1": "1",
                "location": "Tumaco",
                "latitude": 1.8065,
                "longitude": -78.7649,
                "geo_precision": 1,
                "admin1": "Narino",
                "admin2": "Tumaco",
                "civilian_targeting": "Civilian targeting",
                "fatalities": 1,
                "notes": "Asesinato de lider social en zona rural de Tumaco",
                "source": "Indepaz",
                "tags": "",
            },
            {
                "event_id": "ACL003",
                "event_date": "2026-03-15",
                "event_type": "Protests",
                "sub_event_type": "Protest with intervention",
                "disorder_type": "Demonstrations",
                "actor1": "Labor Groups (Colombia)",
                "actor2": "Police (Colombia)",
                "assoc_actor_1": "",
                "assoc_actor_2": "ESMAD",
                "inter1": "6",
                "location": "Cali",
                "latitude": 3.4516,
                "longitude": -76.532,
                "geo_precision": 1,
                "admin1": "Valle del Cauca",
                "admin2": "Cali",
                "civilian_targeting": "",
                "fatalities": 0,
                "notes": "ESMAD interviene protesta social de trabajadores en centro de Cali",
                "source": "El Pais Cali",
                "tags": "",
            },
            {
                "event_id": "ACL004",
                "event_date": "2026-03-18",
                "event_type": "Protests",
                "sub_event_type": "Peaceful protest",
                "disorder_type": "Demonstrations",
                "actor1": "Indigenous Groups (Colombia)",
                "actor2": "",
                "assoc_actor_1": "CRIC",
                "assoc_actor_2": "",
                "inter1": "6",
                "location": "Popayan",
                "latitude": 2.4419,
                "longitude": -76.6063,
                "geo_precision": 1,
                "admin1": "Cauca",
                "admin2": "Popayan",
                "civilian_targeting": "",
                "fatalities": 0,
                "notes": "Minga indigena exige cumplimiento de acuerdos con gobierno antes de elecciones",
                "source": "CRIC",
                "tags": "elecciones",
            },
            {
                "event_id": "ACL005",
                "event_date": "2026-03-20",
                "event_type": "Violence against civilians",
                "sub_event_type": "Attack",
                "disorder_type": "Political violence",
                "actor1": "FARC-EP Dissidents",
                "actor2": "Civilians (Colombia)",
                "assoc_actor_1": "",
                "assoc_actor_2": "",
                "inter1": "1",
                "location": "Arauca",
                "latitude": 7.0847,
                "longitude": -70.7592,
                "geo_precision": 1,
                "admin1": "Arauca",
                "admin2": "Arauca",
                "civilian_targeting": "Civilian targeting",
                "fatalities": 2,
                "notes": "Ataque contra lideres sociales en zona de conflicto armado",
                "source": "Defensoria del Pueblo",
                "tags": "",
            },
            {
                "event_id": "ACL006",
                "event_date": "2026-03-22",
                "event_type": "Strategic developments",
                "sub_event_type": "Looting/property destruction",
                "disorder_type": "Demonstrations",
                "actor1": "Protesters (Colombia)",
                "actor2": "",
                "assoc_actor_1": "",
                "assoc_actor_2": "",
                "inter1": "6",
                "location": "Medellin",
                "latitude": 6.2442,
                "longitude": -75.5812,
                "geo_precision": 1,
                "admin1": "Antioquia",
                "admin2": "Medellin",
                "civilian_targeting": "",
                "fatalities": 0,
                "notes": "Marcha de defensores de derechos humanos frente al Concejo de Medellin",
                "source": "El Colombiano",
                "tags": "",
            },
            {
                "event_id": "ACL007",
                "event_date": "2026-03-25",
                "event_type": "Violence against civilians",
                "sub_event_type": "Attack",
                "disorder_type": "Political violence",
                "actor1": "Unidentified Armed Group",
                "actor2": "Civilians (Colombia)",
                "assoc_actor_1": "",
                "assoc_actor_2": "",
                "inter1": "1",
                "location": "Buenaventura",
                "latitude": 3.8801,
                "longitude": -77.0311,
                "geo_precision": 1,
                "admin1": "Valle del Cauca",
                "admin2": "Buenaventura",
                "civilian_targeting": "Civilian targeting",
                "fatalities": 0,
                "notes": "Amenazas contra periodistas que cubren campana electoral en Buenaventura",
                "source": "FLIP",
                "tags": "",
            },
        ]
    )
    print(f"Loaded {len(df_acled)} synthetic sample events")

In [ ]:
from __future__ import annotations

# Quick schema inspection
if not df_acled.empty:
    print(f"Shape: {df_acled.shape}")
    print(f"Columns: {df_acled.columns.tolist()}")
    print()
    display(df_acled.head(3))
else:
    print("No ACLED data collected — check credentials and API availability")

## 🔍 Exploratory Analysis

Before saving, we inspect the collected events to understand the distribution across
event types, geography, and time. This shapes how we interpret the data in the analysis
notebooks (3-analysis/) and informs the CIVICUS early-warning dashboard.

In [ ]:
from __future__ import annotations

if not df_acled.empty:
    print("=" * 55)
    print("EVENT TYPE DISTRIBUTION")
    print("=" * 55)
    print(df_acled["event_type"].value_counts().to_string())

    print("\n" + "=" * 55)
    print("SUB-EVENT TYPE DISTRIBUTION")
    print("=" * 55)
    print(df_acled["sub_event_type"].value_counts().to_string())

    print("\n" + "=" * 55)
    print("GEOGRAPHIC SPREAD (by department / admin1)")
    print("=" * 55)
    print(df_acled["admin1"].value_counts().to_string())

    print("\n" + "=" * 55)
    print("TEMPORAL DISTRIBUTION")
    print("=" * 55)
    print(df_acled["event_date"].value_counts().sort_index().to_string())

    print("\n" + "=" * 55)
    print("FATALITIES SUMMARY")
    print("=" * 55)
    total_fatalities = df_acled["fatalities"].sum()
    events_with_fatalities = (df_acled["fatalities"] > 0).sum()
    print(f"  Total fatalities: {total_fatalities}")
    print(f"  Events with fatalities: {events_with_fatalities}/{len(df_acled)}")
    print(df_acled["fatalities"].describe().to_string())

    print("\n" + "=" * 55)
    print("TOP ACTORS (actor1)")
    print("=" * 55)
    print(df_acled["actor1"].value_counts().head(10).to_string())
else:
    print("No data available for analysis")

In [ ]:
from __future__ import annotations

# Visualize event type breakdown and geographic distribution
# These two charts are the core visual output for CIVICUS reviewers
if not df_acled.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: Event type distribution
    event_counts = df_acled["event_type"].value_counts()
    event_counts.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="white")
    axes[0].set_title("ACLED Events by Type\n(Colombia, election monitoring scope)", fontsize=12)
    axes[0].set_xlabel("Event Type")
    axes[0].set_ylabel("Count")
    axes[0].tick_params(axis="x", rotation=40)

    # Right: Geographic distribution by department
    geo_counts = df_acled["admin1"].value_counts()
    geo_counts.plot(kind="barh", ax=axes[1], color="coral", edgecolor="white")
    axes[1].set_title("ACLED Events by Department\n(most affected regions)", fontsize=12)
    axes[1].set_xlabel("Count")
    axes[1].invert_yaxis()  # Highest count at top

    plt.suptitle(
        f"ACLED Colombia — {start_date} to {end_date}",
        fontsize=13,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()
else:
    print("No data to visualize")

## 💾 Save to Raw Layer

Save collected events to `data/01_raw/acled/` as a dated JSON file.
The raw layer is immutable — we never modify files here, only add new dated snapshots.

In [ ]:
from __future__ import annotations

# today_str was set in the configuration cell (system clock date for the filename)
output_dir = Path("../../data/01_raw/acled")
output_dir.mkdir(parents=True, exist_ok=True)

if not df_acled.empty:
    output_path = output_dir / f"acled_events_{today_str}.json"
    records = df_acled.to_dict(orient="records")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2, default=str)
    print(f"Saved {len(records)} events to {output_path}")
else:
    print("No ACLED data to save")

## 📊 Collection Summary

In [ ]:
from __future__ import annotations

print("=" * 60)
print("ACLED COLLECTION SUMMARY")
print("=" * 60)

total_events = len(df_acled) if not df_acled.empty else 0
print(f"Date range queried:  {start_date} to {end_date}")
print(f"Events collected:    {total_events}")
print(f"Credentials used:    {'Live API' if has_acled_credentials else 'Synthetic sample'}")

if not df_acled.empty:
    print(f"Actual date range:   {df_acled['event_date'].min()} to {df_acled['event_date'].max()}")
    print(f"Unique departments:  {df_acled['admin1'].nunique()}")
    print(f"Unique actors:       {df_acled['actor1'].nunique()}")
    print(f"Total fatalities:    {df_acled['fatalities'].sum()}")

    print("\nEvent type breakdown:")
    for event_type, count in df_acled["event_type"].value_counts().items():
        print(f"  {event_type:<35} {count}")

    print("\nGeographic distribution (top 5 departments):")
    for dept, count in df_acled["admin1"].value_counts().head(5).items():
        print(f"  {dept:<30} {count}")

print("\nOutput file:")
print(f"  data/01_raw/acled/acled_events_{today_str}.json")

## 💡 Notes & Next Steps

**Research context:**
- ACLED provides the **ground-truth offline baseline** for our online sentiment analysis.
  When we detect a spike in negative electoral sentiment in RSS/GDELT, ACLED can tell us
  whether a real-world event (protest, attack on a social leader) triggered it.
- Conversely, an online discourse spike BEFORE an ACLED event may be an early warning signal —
  this is the core insight CIVICUS Thematic Area 3 (Early Warning Indicators) needs.

**Data quality observations:**
- ACLED has a reporting lag of days to weeks for some event types (rural events)
- The keyword post-filter narrows to research scope but may exclude relevant events
  whose notes don't contain our specific terms — review excluded events periodically
- `geo_precision` field indicates location accuracy: 1=exact, 2=district, 3=admin region

**Coverage gaps:**
- Rural events in Chocó, Meta, Guaviare are systematically underreported
- ACLED relies on media sources — events not covered by media are missed

**Next steps:**
1. Normalize to `data/02_intermediate/` with common schema (shared with RSS/GDELT)
2. Use geocoded coordinates for geographic overlay in analysis summary (notebook 4-output/02)
3. Build temporal correlation between online sentiment (RSS) and offline events (ACLED)
4. Flag departments with both high ACLED violence AND high negative online sentiment

**Scheduling:**
- ACLED updates weekly; run `make collect_acled` weekly or as needed
- Increase `lookback_days` if running after a gap to backfill missing events

## 📖 References

- ACLED: https://acleddata.com
- ACLED API documentation: https://acleddata.com/api-documentation/getting-started
- ACLED Colombia dashboard: https://acleddata.com/dashboard/#/dashboard/CO
- CIVICUS Monitor Colombia: https://monitor.civicus.org/country/colombia/
- Indepaz (Colombia social leaders monitoring): https://indepaz.org.co
- FLIP (Colombian press freedom): https://flip.org.co